# Unused models

These **Generalized Estimating Equation (GEE)** benchmarks live outside [`3 fatigue_modeling.ipynb`](3%20fatigue_modeling.ipynb).

## Why unused?

The main notebook targets **prediction accuracy** (held-out test MAE). GEE is designed for **longitudinal inference**: coefficient estimates with **statistically valid standard errors and p-values after accounting for within-cluster correlation** (repeated daily rows within each participant-interval). That is valuable for interpretability and hypothesis testing, not for ranking predictors by test error.

On this dataset both GEE models land mid-pack (~1.27 test MAE) versus stronger tree and history-feature models (~0.9–1.2 MAE). They stay here for reference and optional re-runs, not the primary accuracy comparison.

Same participant-level split, GroupKFold CV, and Optuna tuning (`GEE_OPTUNA_TRIALS=10`) as the main notebook.


In [ ]:
%pip install -q -r ../../requirements.txt


In [ ]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

from modeling.config import DATA_PATH, GEE_OPTUNA_TRIALS, N_CV_FOLDS
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model
from modeling.summaries import collect_summaries


## Load data and split


In [ ]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


In [ ]:
ordinal_results = []
ordinal_best_params = {}


## GEE models


### `gee_gaussian` (ordinal regression)

**What it does:** Fits a **Gaussian GEE** on `fatigue_num`, treating the ordinal score as continuous. Uses **autoregressive working correlation** within each cluster (participant × `study_interval`, rows sorted by `day_in_study`). Includes `study_interval` as a covariate plus the 17 base daily features. Optuna tunes `maxiter`; the predicted mean is clipped to [0, 5].

**Why unused:** Inference-oriented (valid SEs/p-values under correlation). Not prioritized while we optimize prediction accuracy only.


In [ ]:
_name = 'gee_gaussian'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=GEE_OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'best params: {_params}')
print('GEE cluster = one participant-interval (id × study_interval); summary cluster sizes should be ~90 days per interval.')
print(_result['summary'])
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


### `gee_ordinal` (ordinal classification)

**What it does:** Fits **five cumulative-threshold Binomial GEE** models (P(y > k) for k = 0…4) with autoregressive working correlation (Exchangeable fallback when AR is unstable). Clustered by participant-interval; uses the 17 base daily features only (wave captured by clustering, not a `study_interval` covariate). Class probabilities from threshold survival are combined and clipped to [0, 5].

**Why unused:** Same inference vs accuracy tradeoff as `gee_gaussian`; kept for reference, not the main leaderboard.


In [ ]:
_name = 'gee_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=GEE_OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'best params: {_params}')
print('GEE cluster = one participant-interval (id × study_interval); summary cluster sizes should be ~90 days per interval.')
print(_result['summary'])
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


## Summary


In [ ]:
_, test_summary = collect_summaries(ordinal_results)
display(test_summary.sort_values('test_mae'))
